# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Note: dataset.metadata is an object, not a dictionary. Use dot notation.
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets (`@id` values), as well as their fields and columns.

All references to dataset entities (record sets, fields, columns) will use their `@id` throughout this notebook as per best practice.

In [ ]:
# Use the Croissant API to inspect available record sets and their schema
print("Available Record Sets and Fields:\n-----------------------------")
record_set_ids = []
for record_set in dataset.record_sets:
    print(f"Record set @id: {record_set['@id']}")
    record_set_ids.append(record_set['@id'])
    if 'field' in record_set:
        field_objs = record_set['field']
        if not isinstance(field_objs, list):
            field_objs = [field_objs]
        print("   Fields:")
        for field in field_objs:
            if isinstance(field, dict):
                print(f"      - {field['@id']} (name: {field.get('name','')}, type: {field.get('@type','')})")
            else:
                print(f"      - {field}")
    print()
if not record_set_ids:
    print("No record sets listed in the metadata. Fetching via dataset.record_sets property...")
    # even if empty in top-level, mlcroissant should provide record sets from datafiles
    record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print(f"Record set `@id`s: {record_set_ids}")

# Display a preview of records for each record set
print("\nRecord Examples:")
for rs_id in record_set_ids:
    print(f"\nSample records from record set '{rs_id}':")
    try:
        gen = dataset.records(record_set=rs_id)
        for i, rec in enumerate(gen):
            print(f"  Record {i}: {rec}")
            if i==2:
                break
    except Exception as e:
        print(f"  Could not load records for '{rs_id}': {e}")


## 3. Data Extraction
Load data for the main tabular record set(s) into DataFrames.

All references to fields and record sets use their `@id`.

> **Note:** Record set `@id`s above can be substituted below depending on which data you want to extract.

In [ ]:
# List of detected record set `@id`s
record_sets = record_set_ids
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"\nLoaded DataFrame for: {record_set_id}")
            print(f"Columns: {df.columns.tolist()}")
            print(df.head(3))
        else:
            print(f"\nNo records found for: {record_set_id}")
    except Exception as e:
        print(f"\nCould not extract DataFrame for {record_set_id}: {e}")

# For demonstration, pick the first available record set with data
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nMain record set being used for EDA: {main_record_set_id}")
    print("\nAvailable columns (fields by @id):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None
    print("No dataframes available for EDA!")

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing to the main record set: examining value ranges, filtering, normalization, grouping, and summarizing fields. 

**References:**
- Record set and field names are referenced by their `@id` as above.
- This section assumes at least one numeric field is present.

> Modify `numeric_field_id` and `group_field_id` according to the columns extracted in the previous cell.

In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # List available columns as candidates for numeric/group fields
    print("Columns in main record set:")
    print(df.columns.tolist())

    # Try to guess a numeric field (e.g. 'Age', 'Interval', etc.)
    # You may overwrite these two variables with the field `@id` you want to analyze.
    numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64] or df[col].apply(lambda x: str(x).replace('.', '', 1).isdigit()).all()]
    if not numeric_fields:
        # Try again: Pick fields with numeric-looking values
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except:
                pass
        numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
    else:
        numeric_field_id = df.columns[0]  # fallback

    print(f"Using numeric field for filtering and normalization: {numeric_field_id}")

    # Choose a threshold for demonstration
    try:
        threshold = float(df[numeric_field_id].mean())
    except:
        threshold = 10

    filtered_df = df.copy()
    try:
        filtered_df = filtered_df[pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') > threshold]
    except:
        print("Could not apply numeric thresholding; skipping filtering.")

    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    col = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
    filtered_df[numeric_field_id + '_normalized'] = (col - col.mean()) / col.std()

    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Try grouping by a likely categorical variable
    possible_group_fields = [col for col in df.columns if col != numeric_field_id and df[col].nunique() < 10]
    group_field_id = possible_group_fields[0] if possible_group_fields else None

    if group_field_id:
        print(f"\nGrouping by field: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count'])
        display(grouped_df.head())
    else:
        print("\nNo suitable grouping field found.")
else:
    print("No data available for EDA section.")

## 5. Visualization
Visualize data distributions and relationships between fields using matplotlib or seaborn.

Again, use explicit `@id` field references from the DataFrame columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id:
    df = dataframes[main_record_set_id]
    if numeric_field_id in df.columns:
        plt.figure(figsize=(6,3))
        sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), kde=True, bins=15)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.tight_layout()
        plt.show()

    if group_field_id and numeric_field_id in df.columns:
        plt.figure(figsize=(7,3))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, whis=1.5)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("No data available to plot.")

## 6. Conclusion
This notebook demonstrated how to access and analyze the FAIR² dataset's tabular record sets using the `mlcroissant` library, referencing all data schema elements by their `@id` field.

- We loaded dataset metadata and explored available record sets and fields.
- Data was extracted to pandas DataFrames, using only `@id` references for columns and record sets.
- Typical EDA steps were shown: filtering on a numeric field, normalization, grouping, and quick visualizations.

You may extend this analysis by linking further fields using their `@id` or by delving deeper into the domain-specific structure of the FAIR² Croissant package.

For more advanced data integration or ML experiments, always use `@id` for referencing entities to ensure robustness and metadata-compatibility.